### Links and Resources
| Method / Class | Description |
|----------------|-------------|
| [DeltaTable](https://docs.delta.io/latest/api/python/spark/index.html) | Entry point for Delta Lake table operations |
| [DeltaTable.forName()](https://docs.delta.io/latest/api/python/spark/index.html) | Loads a Delta table by its Unity Catalog name (`catalog.schema.table`) |
| [DeltaTable.forPath()](https://docs.delta.io/latest/api/python/spark/index.html) | Loads a Delta table by its storage path (DBFS or Volume path) |
| [DeltaTable.toDF()](https://docs.delta.io/latest/api/python/spark/index.html) | Returns the Delta table as a Spark DataFrame |
| [DeltaTable.history()](https://docs.delta.io/latest/api/python/spark/index.html) | Returns a DataFrame with the transaction log history of the table |
| [DeltaTable.detail()](https://docs.delta.io/latest/api/python/spark/index.html) | Returns metadata about the Delta table (size, partitioning, format, etc.) |
| [DeltaTable.delete()](https://docs.delta.io/latest/api/python/spark/index.html) | Deletes rows matching a condition (Python API equivalent of SQL `DELETE FROM … WHERE`) |
| [DELETE FROM (SQL)](https://docs.databricks.com/en/sql/language-manual/delta-delete-from.html) | SQL syntax for deleting rows from a Delta table |
| [Time Travel](https://docs.databricks.com/en/delta/history.html) | Read earlier versions of a Delta table with `VERSION AS OF` or `TIMESTAMP AS OF` |

In [ ]:
import os
if not os.getenv("DATABRICKS_RUNTIME_VERSION"):
    from spark_setup import spark, display  # local run through Databricks Connect

DATA_ROOT = "/Volumes/population_metrics/landing/datasets"

# Delta Table Introduction

**Delta Lake** is an open-source storage layer that brings ACID transactions, schema enforcement, and time travel to Apache Spark workloads. Every Delta table is backed by a `_delta_log/` transaction log that records every operation applied to the table.

The `DeltaTable` class is the main Python entry point for interacting with Delta tables programmatically. There are two ways to load an existing Delta table:

| Method | When to use |
|--------|-------------|
| `DeltaTable.forName(spark, "catalog.schema.table")` | Table is registered in Unity Catalog (or the Hive metastore) |
| `DeltaTable.forPath(spark, "/path/to/delta/table")` | You know the physical storage path of the Delta table |

### Importing DeltaTable

`DeltaTable` lives in the `delta.tables` package, which is pre-installed on every Databricks cluster with Delta Lake support.

In [ ]:
from delta.tables import DeltaTable

### `DeltaTable.forName()` — Load by Catalog Name

`forName` resolves the table through Unity Catalog using the three-part identifier `catalog.schema.table`.  
This is the preferred approach when tables are managed by Unity Catalog because it is location-independent — you do not need to know where the data is physically stored.

In [ ]:
# Load the country_regions managed table from Unity Catalog
# Catalog: population_metrics | Schema: default | Table: country_regions
dt_by_name = DeltaTable.forName(spark, "population_metrics.default.country_regions")

print(type(dt_by_name))  # delta.tables.DeltaTable

# Convert to a regular Spark DataFrame for querying
df_by_name = dt_by_name.toDF()
display(df_by_name)

### Inspect Schema and Metadata via `forName`

`detail()` returns a single-row DataFrame with table-level metadata: location, format, creation time, number of files, and more.

In [ ]:
# Print the schema of the underlying DataFrame
print("Schema:")
df_by_name.printSchema()

# detail() gives metadata about the Delta table itself
print("\nTable detail:")
display(dt_by_name.detail())

### Transaction History via `forName`

`history()` reads the `_delta_log/` transaction log and returns one row per commit. Each row shows the operation type (`WRITE`, `CREATE TABLE`, `MERGE`, etc.), timestamp, user, and Spark version.

In [ ]:
# Show the last 10 operations on the country_regions table
display(dt_by_name.history(10).select("version", "timestamp", "operation", "operationParameters"))

### `DeltaTable.forPath()` — Load by Storage Path

`forPath` bypasses Unity Catalog and reads the Delta table directly from its storage location.  
This is useful when:
- The table is not registered in a catalog (external or legacy tables).
- You are working with a path in a Unity Catalog Volume or DBFS.
- You need to access a specific physical location regardless of catalog registration.

The data at the countries_population CSV path will first be written as a **Delta table** to a location in the Volume, and then accessed with `forPath`.

In [ ]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

csv_path = (
    f"{DATA_ROOT}/countries_dataset/"
    "countries_dataset/csv_data/countries_population/countries_population.csv"
)

schema = StructType([
    StructField("COUNTRY_ID",    IntegerType(), True),
    StructField("NAME",          StringType(),  True),
    StructField("NATIONALITY",   StringType(),  True),
    StructField("CODE",          StringType(),  True),
    StructField("ISO_ALPHA3",    StringType(),  True),
    StructField("CAPITAL",       StringType(),  True),
    StructField("POPULATION",    IntegerType(), True),
    StructField("AREA_KM2",      DoubleType(),  True),
    StructField("REGION_ID",     IntegerType(), True),
    StructField("SUB_REGION_ID", IntegerType(), True),
])

# Read the source CSV
df_csv = spark.read.format("csv").schema(schema).option("header", True).load(csv_path)

print(f"Rows read from CSV: {df_csv.count()}")
display(df_csv)

### Write the CSV Data as a Delta Table to a Volume Path

Delta tables written to a Volume path are not automatically registered in Unity Catalog — they exist only at the physical path.  
This is exactly the scenario where `forPath` is the right tool.

In [ ]:
# Target path for the Delta table inside the Volume
delta_path = f"{DATA_ROOT}/countries_dataset/countries_dataset/delta/countries_population_delta"

# Write as Delta format (overwrite if it already exists)
df_csv.write.format("delta").mode("overwrite").save(delta_path)

print(f"Delta table written to: {delta_path}")

dt = DeltaTable.forPath(spark, delta_path)

type(dt)

#display(dt.toDF())

### Load the Delta Table via `forPath`

Now that a Delta table exists at the Volume path, `DeltaTable.forPath` can load it directly without any catalog registration.

In [ ]:
# Load the Delta table by its physical storage path
dt_by_path = DeltaTable.forPath(spark, delta_path)

print(type(dt_by_path))  # delta.tables.DeltaTable

# Convert to DataFrame for querying
df_by_path = dt_by_path.toDF()
display(df_by_path)

### Inspect Schema and Metadata via `forPath`

In [ ]:
# Schema of the Delta table loaded by path
print("Schema:")
df_by_path.printSchema()

# Metadata: size, location, format details
print("\nTable detail:")
display(dt_by_path.detail())

### Transaction History via `forPath`

The transaction log works identically whether the table was loaded by name or by path.

In [ ]:
display(dt_by_path.history().select("version", "timestamp", "operation", "operationParameters"))

### Summary: `forName` vs `forPath`

| | `forName` | `forPath` |
|-|-----------|-----------|
| **Input** | `"catalog.schema.table"` | Physical storage path |
| **Requires catalog registration** | Yes | No |
| **Location-independent** | Yes | No |
| **Works with external/unregistered tables** | No | Yes |
| **Preferred for Unity Catalog managed tables** | Yes | No |

Both methods return a `DeltaTable` object with the same API: `toDF()`, `detail()`, `history()`, `update()`, `delete()`, `merge()`, and more.

# Deleting Records from a Delta Table

Delta Lake implements deletes using its **transaction log** — a row is never physically removed immediately. Instead, the commit records which rows are logically deleted (using **Deletion Vectors** when enabled, or by rewriting Parquet files when not). This makes deletes:

- **ACID-compliant** — concurrent readers see a consistent snapshot.
- **Time-travel-friendly** — earlier versions of the table still contain the deleted rows.
- **Audit-ready** — every `DELETE` appears as a versioned entry in `history()`.

There are two equivalent syntaxes to delete rows:

| Approach | API |
|----------|-----|
| Python (programmatic) | `DeltaTable.delete(condition)` |
| SQL | `DELETE FROM catalog.schema.table WHERE condition` |

### Dataset used in this section

A small **product inventory** table (`population_metrics.default.product_inventory`) will be created fresh, registered in Unity Catalog, and used to demonstrate every deletion variant.

### Create the Dataset and Register in Unity Catalog

The table is created with `CREATE OR REPLACE TABLE` so re-running the notebook always starts from a clean state.  
`population_metrics.default.product_inventory` is a **managed** Delta table — Unity Catalog owns its lifecycle and storage location.

In [ ]:
from pyspark.sql import Row

TABLE = "population_metrics.default.product_inventory"

# 12 products across three categories
rows = [
    Row(product_id=1,  name="Laptop Pro 15",      category="Electronics", price=1299.99, in_stock=True),
    Row(product_id=2,  name="Wireless Mouse",      category="Electronics", price=25.49,  in_stock=True),
    Row(product_id=3,  name="USB-C Hub",           category="Electronics", price=39.99,  in_stock=False),
    Row(product_id=4,  name="Standing Desk",       category="Furniture",   price=499.00, in_stock=True),
    Row(product_id=5,  name="Ergonomic Chair",     category="Furniture",   price=349.00, in_stock=True),
    Row(product_id=6,  name="Monitor Arm",         category="Furniture",   price=89.95,  in_stock=False),
    Row(product_id=7,  name="Notebook (A5)",       category="Stationery",  price=4.99,   in_stock=True),
    Row(product_id=8,  name="Ballpoint Pen 10pk",  category="Stationery",  price=2.49,   in_stock=True),
    Row(product_id=9,  name="Sticky Notes 5pk",    category="Stationery",  price=3.79,   in_stock=True),
    Row(product_id=10, name="Whiteboard Marker",   category="Stationery",  price=5.99,   in_stock=False),
    Row(product_id=11, name="Coffee Mug",          category="Kitchen",     price=12.00,  in_stock=True),
    Row(product_id=12, name="Desk Lamp LED",       category="Electronics", price=45.00,  in_stock=True),
]

df_products = spark.createDataFrame(rows)

# Persist as a managed Delta table in Unity Catalog (overwrite for idempotency)
df_products.write.format("delta").mode("overwrite").saveAsTable(TABLE)

print(f"Table '{TABLE}' created with {df_products.count()} rows.")
display(spark.table(TABLE))

## 1. Delete a Single Record — Python API

`DeltaTable.delete(condition)` accepts a SQL-string condition or a Spark Column expression.  
The condition behaves exactly like a `WHERE` clause: only rows that evaluate to `true` are removed.

**Scenario:** product `#3` ("USB-C Hub") has been discontinued — remove it by its primary key.

In [ ]:
dt_inv = DeltaTable.forName(spark, TABLE)

# --- Python API: delete by primary key (single row) ---
dt_inv.delete("product_id = 3")

print("After deleting product_id = 3 ('USB-C Hub'):")
display(spark.table(TABLE).orderBy("product_id"))

## 2. Delete Multiple Records by Condition — Python API

The same `delete()` method works for any predicate.  
You can pass either a **SQL string** (`"column = value"`) or a **Spark Column** (`F.col("column") == value`).

**Scenario:** Remove all products that are out of stock (`in_stock = False`).

In [ ]:
from pyspark.sql import functions as F

# --- Python API: delete using a Spark Column expression ---
# Removes all rows where in_stock is False (products #6 and #10 remain after #3 was deleted above)
dt_inv.delete(F.col("in_stock") == False)

remaining = spark.table(TABLE).orderBy("product_id")
print(f"Rows remaining after removing out-of-stock products: {remaining.count()}")
display(remaining)

## 3. Delete Records — SQL Syntax

`DELETE FROM <table> WHERE <condition>` is the ANSI-standard SQL syntax. In Databricks you can run it with `spark.sql()` or directly in a `%sql` cell.  
It is fully equivalent to the Python API — both produce the same Delta transaction log entry.

**Scenario:** Remove the entire "Kitchen" category (only "Coffee Mug" at this point).

In [ ]:
# --- SQL syntax: DELETE with a category filter ---
spark.sql(f"DELETE FROM {TABLE} WHERE category = 'Kitchen'")

remaining = spark.table(TABLE).orderBy("product_id")
print(f"Rows remaining after SQL delete of 'Kitchen' category: {remaining.count()}")
display(remaining)

## 4. Delete All Records — SQL (Full Table Clear)

Omitting the `WHERE` clause from `DELETE FROM` removes every row from the table.  
The table structure (schema, Delta log, metadata) is preserved — only the data is gone.  
This is softer than `DROP TABLE` or `TRUNCATE TABLE`: the Delta log survives and time travel back to previous versions is still possible.

In [ ]:
# --- SQL syntax: DELETE without WHERE — removes every row ---
spark.sql(f"DELETE FROM {TABLE}")

remaining = spark.table(TABLE)
print(f"Rows remaining after full delete: {remaining.count()}")  # expected: 0
display(remaining)

## 5. Audit Trail — Transaction History After Deletions

Every `delete()` call or SQL `DELETE` statement writes a new commit to the `_delta_log/`.  
`history()` exposes the full audit trail: who deleted what, when, and with which predicate.

In [ ]:
# Reload the DeltaTable handle (after all the deletes above)
dt_inv = DeltaTable.forName(spark, TABLE)

# Show all versions in chronological order (oldest first)
display(
    dt_inv.history()
    .select("version", "timestamp", "operation", "operationParameters")
    .orderBy("version")
)

## 6. Time Travel — Reading Data Before a Delete

Because Delta Lake retains all historical Parquet files until `VACUUM` is run, you can read the table **as it was at any earlier version**.  
This is the safety net for accidental deletes: the data is still there — just not visible in the current version.

| Time-travel syntax | Description |
|--------------------|-------------|
| `VERSION AS OF n` | Read the table at a specific version number |
| `TIMESTAMP AS OF 'ts'` | Read the table as of a point in time |

> **Note:** historical files are kept according to the table's `delta.deletedFileRetentionDuration` property (default 7 days). `VACUUM` permanently removes them.

In [ ]:
# --- Read the original full dataset (version 0 — before any deletes) ---
df_v0 = spark.sql(f"SELECT * FROM {TABLE} VERSION AS OF 0 ORDER BY product_id")

print(f"Version 0 row count (original): {df_v0.count()}")
display(df_v0)

# --- Read after the first single-row delete (version 1) ---
df_v1 = spark.sql(f"SELECT * FROM {TABLE} VERSION AS OF 1 ORDER BY product_id")
print(f"\nVersion 1 row count (after deleting product_id=3): {df_v1.count()}")
display(df_v1)

## 7. DataFrame Filter + Overwrite (Anti-pattern)

Before `DeltaTable.delete()` existed, the common workaround was to:
1. Read the whole table into a DataFrame.
2. Filter out the unwanted rows.
3. Overwrite the table with the filtered DataFrame.

This still works, but it is a **full-table rewrite** regardless of how many rows you actually want to remove.

In [ ]:
from pyspark.sql import Row

SMALL_TABLE = "population_metrics.default.small_products"

# Step 0 — create a small 5-row table to demonstrate the pattern
rows = [
    Row(product_id=1, name="Laptop",   price=999.0),
    Row(product_id=2, name="Mouse",    price=25.0),
    Row(product_id=3, name="Keyboard", price=49.0),
    Row(product_id=4, name="Monitor",  price=299.0),
    Row(product_id=5, name="Webcam",   price=79.0),
]
spark.createDataFrame(rows).write.format("delta").mode("overwrite").saveAsTable(SMALL_TABLE)

print("Initial table:")
display(spark.table(SMALL_TABLE).orderBy("product_id"))

# Step 1 — read the full table into a DataFrame
df_full = spark.table(SMALL_TABLE)

# Step 2 — filter OUT the row to delete (keep everything except product_id = 3)
df_filtered = df_full.filter("product_id != 3")

print("Filtered DataFrame (product_id = 3 excluded):")
display(df_filtered.orderBy("product_id"))

# Step 3 — overwrite the table with the filtered DataFrame
df_filtered.write.format("delta").mode("overwrite").saveAsTable(SMALL_TABLE)

print("Table after overwrite:")
display(spark.table(SMALL_TABLE).orderBy("product_id"))

### Why this approach is inefficient

Even though only one row was removed, **every row was read and every Parquet file was rewritten**:

| Cost | `DeltaTable.delete()` / SQL `DELETE` | DataFrame filter + overwrite |
|------|--------------------------------------|------------------------------|
| **Files read** | Only the files that contain matching rows | All files in the table |
| **Files written** | Only the affected files (or a deletion vector) | All files in the table |
| **Data shuffled** | Minimal | Full table scan + full write |
| **Concurrent readers blocked** | No — snapshot isolation | Potentially — full overwrite |
| **History / audit** | `DELETE` operation recorded with predicate | `WRITE` operation — predicate lost |
| **Schema safety** | Enforced by Delta | Must manually preserve schema |

For a 5-row toy table the difference is invisible. On a table with millions of rows across hundreds of Parquet files, the overwrite approach reads and rewrites **gigabytes of data** to remove a single row. Use `DeltaTable.delete()` or SQL `DELETE FROM` instead.

## Summary: Deletion Techniques

| Technique | Syntax | Deletes |
|-----------|--------|---------|
| **Python — single row by PK** | `dt.delete("product_id = 3")` | Exactly one row |
| **Python — multi-row condition** | `dt.delete(F.col("in_stock") == False)` | All matching rows |
| **SQL — category filter** | `DELETE FROM t WHERE category = 'Kitchen'` | All rows in category |
| **SQL — full table clear** | `DELETE FROM t` | Every row (table shell preserved) |

All four techniques:
- Write a `DELETE` commit to the Delta transaction log.
- Appear in `history()` with the predicate used.
- Leave earlier versions readable via `VERSION AS OF n` (until `VACUUM` runs).
- Do **not** drop the table or its schema.